# Detection Walkthrough

This notebook demonstrates how the AI-Support Fabric detection engine identifies anomalies in telemetry data.

## Detection Architecture

The detection engine consists of multiple specialized detectors:

1. **LatencySpikeDetector**: Identifies unusually slow requests
2. **ConfigDriftDetector**: Detects configuration changes from baseline
3. **AuthFailureDetector**: Identifies potential authentication attacks

Each detector analyzes telemetry data and generates `Finding` objects when anomalies are detected.

In [ ]:
# Setup
import requests
import json
from datetime import datetime
import time

GATEWAY_URL = "http://localhost:8080"

def pretty_print(data):
    """Pretty print JSON data"""
    print(json.dumps(data, indent=2))

def run_analysis():
    """Trigger AI analysis"""
    response = requests.post(f"{GATEWAY_URL}/api/run-analysis")
    return response.json()

## Scenario 1: Latency Spike Detection

Let's inject telemetry showing slow requests and see how the detector identifies the issue.

In [ ]:
# Generate slow request logs
print("Generating slow request telemetry...")

for i in range(10):
    log_data = {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "service": "api-service",
        "level": "WARNING",
        "message": "Request took longer than expected",
        "request_id": f"slow-req-{i}",
        "duration_ms": 5000 + (i * 500),  # 5-9.5 seconds
        "expected_duration_ms": 200
    }
    
    requests.post(f"{GATEWAY_URL}/api/telemetry/logs", json=log_data)

print("✓ Slow request telemetry generated")

In [ ]:
# Run detection
print("Running AI analysis...")
result = run_analysis()

print(f"\nAnalysis complete. Found {result['result']['findings_count']} finding(s)")

# Display findings
for finding in result['result']['findings']:
    print(f"\n{'='*60}")
    print(f"Finding: {finding['title']}")
    print(f"Severity: {finding['severity']}")
    print(f"Description: {finding['description']}")
    print(f"\nRecommendations:")
    for i, rec in enumerate(finding['recommendations'], 1):
        print(f"  {i}. {rec}")

## Scenario 2: Configuration Drift Detection

Now let's simulate a configuration change and see how drift detection works.

In [ ]:
# Send configuration with drift
print("Sending configuration with security drift...")

config_data = {
    "timestamp": datetime.utcnow().isoformat() + "Z",
    "service": "api-service",
    "config_version": "1.0.1-drift",
    "configuration": {
        "debug_mode": True,  # DRIFT: Should be False
        "log_level": "DEBUG",  # DRIFT: Should be INFO
        "max_connections": 100,
        "timeout_seconds": 30,
        "rate_limit": 1000,
        "enable_auth": True,
        "tls_enabled": True
    },
    "changed_by": "unknown",
    "change_reason": "Unauthorized change detected"
}

response = requests.post(f"{GATEWAY_URL}/api/telemetry/config", json=config_data)
print("✓ Configuration drift telemetry sent")

In [ ]:
# Run detection again
print("Running AI analysis...")
time.sleep(1)  # Brief pause
result = run_analysis()

print(f"\nAnalysis complete. Found {result['result']['findings_count']} finding(s)")

# Display config drift findings
for finding in result['result']['findings']:
    if 'config' in finding['title'].lower():
        print(f"\n{'='*60}")
        print(f"Finding: {finding['title']}")
        print(f"Severity: {finding['severity']}")
        print(f"Description: {finding['description']}")
        print(f"\nEvidence:")
        pretty_print(finding['evidence'][:1])  # Show first evidence

## Scenario 3: Authentication Failure Storm

Let's simulate a potential brute-force attack scenario.

In [ ]:
# Generate authentication failure logs
print("Generating authentication failure telemetry...")

for i in range(20):
    log_data = {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "service": "auth-service",
        "level": "ERROR",
        "message": "Authentication failed",
        "request_id": f"auth-fail-{i}",
        "user": f"user_{i % 3}",  # 3 users being targeted
        "reason": "Invalid credentials",
        "source_ip": f"192.168.1.{100 + (i % 2)}"  # 2 IPs attacking
    }
    
    requests.post(f"{GATEWAY_URL}/api/telemetry/logs", json=log_data)

print("✓ Authentication failure telemetry generated")

In [ ]:
# Run detection
print("Running AI analysis...")
time.sleep(1)
result = run_analysis()

print(f"\nAnalysis complete. Found {result['result']['findings_count']} finding(s)")

# Display auth failure findings
for finding in result['result']['findings']:
    if 'auth' in finding['title'].lower():
        print(f"\n{'='*60}")
        print(f"Finding: {finding['title']}")
        print(f"Severity: {finding['severity']}")
        print(f"Description: {finding['description']}")
        print(f"\nEvidence Summary:")
        evidence = finding.get('evidence', {})
        if isinstance(evidence, dict):
            print(f"  Total Failures: {evidence.get('total_failures')}")
            print(f"  Unique Users: {evidence.get('unique_users')}")
            print(f"  Unique IPs: {evidence.get('unique_ips')}")
            print(f"  Top IPs: {evidence.get('top_ips', [])}")

## Understanding Detection Logic

### Latency Spike Detector

**Algorithm**:
1. Scan log entries for `duration_ms` field
2. Count requests exceeding threshold (default: 1000ms)
3. If count >= minimum occurrences (default: 3), generate finding
4. Calculate average duration for context

**Threshold Configuration**:
- Threshold: 1000ms
- Minimum occurrences: 3

### Config Drift Detector

**Algorithm**:
1. Compare current config against baseline
2. Identify any differences (drift)
3. Severity based on security impact:
   - CRITICAL: auth/TLS changes
   - MEDIUM: other changes

**Baseline Configuration**:
```python
{
    'debug_mode': False,
    'log_level': 'INFO',
    'enable_auth': True,
    'tls_enabled': True
}
```

### Auth Failure Detector

**Algorithm**:
1. Scan logs for authentication failures
2. Count total failures
3. If count >= threshold (default: 10), generate finding
4. Group by user and IP for analysis
5. Identify patterns indicating brute-force attacks

## Next Steps

Continue to **Notebook 03: Guided Remediation** to learn how the system generates actionable remediation plans for these findings.